# Road Shortest Distance Study

For each active concrete plant, finds the road distance to its nearest coal plant using the OSRM public routing API. Aggregates results by NRMCA region and nationally, then compares to NRMCA-reported truck distances.

**Method:** `scipy.spatial.KDTree` for candidate pre-filtering, OSRM public API for road routing distance.

**Source:** NRMCA truck distance data from Tables C2–C9, *NRMCA Industry Wide LCA Report V3.2* (2023)

## Imports & Constants

In [ ]:
import pandas as pd
import numpy as np
import requests
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import KDTree
import time
import warnings
warnings.filterwarnings('ignore')

N_CANDIDATES   = 5       # nearest coal plants to evaluate per concrete plant
REQUEST_DELAY  = 0.05    # seconds between OSRM API calls
OSRM_BASE      = 'http://router.project-osrm.org/route/v1/driving'
EARTH_RADIUS_MI = 3958.8

## Load Data

In [ ]:
concrete = pd.read_csv('../02_processed_data/active_concrete_plants_ec3.csv')
coal     = pd.read_csv('../02_processed_data/active_coal_plants_US.csv')
coal_valid = coal.dropna(subset=['Latitude', 'Longitude']).copy()

print(f'Concrete plants loaded:             {len(concrete)}')
print(f'Coal plants with valid coordinates: {len(coal_valid)}')

## Assign NRMCA Regions to Concrete Plants

Since the concrete plant CSV contains only coordinates (no state column), we perform a spatial join against US state boundaries from the Census Bureau cartographic boundary file. State abbreviations are then mapped to the 8 NRMCA regions using a hardcoded lookup.

In [ ]:
# NRMCA region → state abbreviation mapping (hardcoded from NRMCA LCA Report V3.2)
NRMCA_REGION_STATES = {
    'Eastern':             ['ME','NH','VT','MA','RI','CT','NY','NJ','PA','DE','MD','VA','WV'],
    'Great Lakes Midwest': ['OH','IN','IL','MI','WI'],
    'North Central':       ['MN','ND','SD','NE','IA'],
    'Pacific Northwest':   ['WA','OR','ID','MT'],
    'Pacific Southwest':   ['CA','NV','AZ'],
    'Rocky Mountains':     ['CO','UT','NM','WY'],
    'South Central':       ['OK','TX','AR','LA','KS','MO'],
    'South Eastern':       ['KY','TN','NC','SC','GA','FL','AL','MS'],
}

STATE_TO_REGION = {
    state: region
    for region, states in NRMCA_REGION_STATES.items()
    for state in states
}

# Download Census Bureau state boundaries once and cache locally
import os
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

STATES_URL   = 'https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_state_500k.zip'
STATES_LOCAL = '../01_raw_data/cb_2022_us_state_500k.zip'

if not os.path.exists(STATES_LOCAL):
    print('Downloading Census Bureau state boundaries...')
    r = requests.get(STATES_URL, timeout=30, verify=False)
    r.raise_for_status()
    with open(STATES_LOCAL, 'wb') as f:
        f.write(r.content)
    print('Saved to', STATES_LOCAL)

states_gdf = gpd.read_file(STATES_LOCAL)
# Filter to NRMCA-covered states only; Census file uses 'STUSPS' for 2-letter abbreviation
us_states = states_gdf[['STUSPS', 'geometry']].copy()
us_states = us_states[us_states['STUSPS'].isin(STATE_TO_REGION)]

# Convert concrete plants to GeoDataFrame
concrete_gdf = gpd.GeoDataFrame(
    concrete,
    geometry=[Point(lon, lat) for lat, lon in zip(concrete['latitude'], concrete['longitude'])],
    crs='EPSG:4326'
)

# Spatial join: assign state abbreviation to each plant
joined = gpd.sjoin(
    concrete_gdf,
    us_states[['STUSPS', 'geometry']].set_geometry('geometry'),
    how='left',
    predicate='within'
)
joined['region'] = joined['STUSPS'].map(STATE_TO_REGION)

# Keep only plants assigned to a region
concrete_regions = joined.dropna(subset=['region'])[['plant_name', 'latitude', 'longitude', 'region']].copy()

print(f'Plants with assigned NRMCA region: {len(concrete_regions)} / {len(concrete)}')
print()
print(concrete_regions['region'].value_counts().sort_index())

## NRMCA Reference Data

Truck distances (miles) from Tables C2–C9, *NRMCA Industry Wide LCA Report V3.2* (2023). Hardcoded for reference in the output table.

In [ ]:
NRMCA_TRUCK_MI = {
    'Eastern':             100.5,
    'Great Lakes Midwest':  66.3,
    'North Central':       132.3,
    'Pacific Northwest':    75.4,
    'Pacific Southwest':    38.8,
    'Rocky Mountains':     159.5,
    'South Central':        55.1,
    'South Eastern':        58.0,
}
NRMCA_NATIONAL_AVG_MI = 61.7

nrmca_df = pd.DataFrame([
    {'Region': region, 'NRMCA Truck Distance (mi)': dist}
    for region, dist in NRMCA_TRUCK_MI.items()
])
nrmca_df

## Find Closest Coal Plant per Concrete Plant (OSRM)

> **Note:** The two cells below (routing loop + save) only need to be re-run if input data has changed. Results are already saved to `02_processed_data/concrete_plant_closest_coal_distances.csv` and loaded automatically in the aggregation step. The routing cell is smart enough to resume from the existing CSV, so only new or missing plants will be processed.

> **Coverage:** The initial run produced distances for 1,026 of 1,035 region-assigned plants (~99%). The remaining ~9 plants received no valid OSRM response after retries, likely due to coordinates in locations with limited road network coverage in the routing engine.

> **Runtime:** Running from scratch takes **over an hour**. Appending only new records will be much faster.

For each concrete plant:
1. KDTree query finds the `N_CANDIDATES` nearest coal plants by straight-line chord distance
2. OSRM API returns the actual road distance for each candidate
3. The minimum road distance is recorded

Uses `ThreadPoolExecutor` with 5 concurrent workers and retry logic (up to 3 attempts with exponential backoff).

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

def to_unit_sphere(lat, lon):
    """Convert lat/lon arrays to unit-sphere XYZ for KDTree distance queries."""
    lat_r = np.radians(lat)
    lon_r = np.radians(lon)
    x = np.cos(lat_r) * np.cos(lon_r)
    y = np.cos(lat_r) * np.sin(lon_r)
    z = np.sin(lat_r)
    return np.column_stack([x, y, z])


def get_road_distance_miles(lat1, lon1, lat2, lon2, retries=3):
    """Return road distance in miles via OSRM, or None after retries."""
    url = f'{OSRM_BASE}/{lon1},{lat1};{lon2},{lat2}'
    for attempt in range(retries):
        try:
            resp = requests.get(url, params={'overview': 'false'}, timeout=15)
            resp.raise_for_status()
            data = resp.json()
            if data.get('code') != 'Ok':
                return None
            return data['routes'][0]['distance'] / 1609.34
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)  # 1s, 2s backoff
    return None


def process_plant(row):
    """Find the minimum road distance from one concrete plant to its N nearest coal plants."""
    plant_xyz = to_unit_sphere(np.array([row['latitude']]), np.array([row['longitude']]))
    _, indices = tree.query(plant_xyz, k=N_CANDIDATES)
    candidates = coal_valid.iloc[indices[0]]

    min_dist = None
    for _, coal_row in candidates.iterrows():
        dist = get_road_distance_miles(
            row['latitude'], row['longitude'],
            coal_row['Latitude'], coal_row['Longitude']
        )
        if dist is not None and (min_dist is None or dist < min_dist):
            min_dist = dist

    return {'plant_name': row['plant_name'], 'region': row['region'], 'closest_coal_mi': min_dist}


# Build KDTree on coal plant unit-sphere coordinates
coal_xyz = to_unit_sphere(coal_valid['Latitude'].values, coal_valid['Longitude'].values)
tree = KDTree(coal_xyz)

# Load any existing results and only process plants that are missing
DISTANCES_CSV = '../02_processed_data/concrete_plant_closest_coal_distances.csv'
if os.path.exists(DISTANCES_CSV):
    existing_df = pd.read_csv(DISTANCES_CSV)
    already_done = set(existing_df['plant_name'])
    print(f'Loaded {len(existing_df)} existing results, resuming from where we left off...')
else:
    existing_df = pd.DataFrame()
    already_done = set()

remaining = concrete_regions[~concrete_regions['plant_name'].isin(already_done)]
print(f'Plants still to process: {len(remaining)} / {len(concrete_regions)}')
print()

if len(remaining) == 0:
    print('All plants already processed.')
    distances_df = existing_df
else:
    rows = [row for _, row in remaining.iterrows()]
    new_results = [None] * len(rows)
    completed = 0

    with ThreadPoolExecutor(max_workers=5) as executor:
        future_to_idx = {executor.submit(process_plant, row): i for i, row in enumerate(rows)}
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            new_results[idx] = future.result()
            completed += 1
            if completed % 100 == 0:
                print(f'  {completed} / {len(rows)} processed...')

    new_df = pd.DataFrame(new_results)
    distances_df = pd.concat([existing_df, new_df], ignore_index=True).dropna(subset=['closest_coal_mi'])
    print(f'\nDone. Total plants with valid distances: {len(distances_df)} / {len(concrete_regions)}')

distances_df.head()

In [ ]:
# Save results so the OSRM routing cell doesn't need to be re-run
distances_df.to_csv('../02_processed_data/concrete_plant_closest_coal_distances.csv', index=False)
print(f'Saved {len(distances_df)} rows to 02_processed_data/concrete_plant_closest_coal_distances.csv')

In [11]:
# Load pre-computed distances from CSV
distances_df = pd.read_csv('../02_processed_data/concrete_plant_closest_coal_distances.csv')
print(f'Loaded {len(distances_df)} plant distances')

# Mean closest coal plant distance per NRMCA region
regional = (
    distances_df.groupby('region')['closest_coal_mi']
    .mean()
    .reset_index()
    .rename(columns={'region': 'Region', 'closest_coal_mi': 'Avg Calculated Distance (mi)'})
)

# National average across all plants
national_avg = distances_df['closest_coal_mi'].mean()

# Merge with NRMCA reference data
summary = nrmca_df.merge(regional, on='Region', how='left')

# Append national average row
national_row = pd.DataFrame([{
    'Region':                    'National Average',
    'NRMCA Truck Distance (mi)': NRMCA_NATIONAL_AVG_MI,
    'Avg Calculated Distance (mi)':  national_avg
}])
summary = pd.concat([summary, national_row], ignore_index=True)

# Display formatted table
summary['NRMCA Truck Distance (mi)']    = summary['NRMCA Truck Distance (mi)'].round(1)
summary['Avg Calculated Distance (mi)'] = summary['Avg Calculated Distance (mi)'].round(1)

summary.style \
    .set_caption('Fly Ash Truck Distance: NRMCA Reported vs. Calculated from Plant Proximity Data') \
    .format({'NRMCA Truck Distance (mi)': '{:.1f}', 'Avg Calculated Distance (mi)': '{:.1f}'}) \
    .hide(axis='index')

Loaded 1026 plant distances


Region,NRMCA Truck Distance (mi),Avg Calculated Distance (mi)
Eastern,100.5,140.1
Great Lakes Midwest,66.3,76.4
North Central,132.3,64.0
Pacific Northwest,75.4,359.4
Pacific Southwest,38.8,473.3
Rocky Mountains,159.5,112.6
South Central,55.1,85.6
South Eastern,58.0,101.4
National Average,61.7,197.3
